# GDP Nowcasting with Dynamic Factor Models

**Goal:** Build a working GDP nowcasting system using real FRED data in under 15 minutes.

**What you'll learn:**
- Download and prepare real-time macroeconomic data
- Estimate a Dynamic Factor Model
- Build a bridge equation linking factors to GDP
- Produce a nowcast with confidence bands
- Decompose nowcast changes (news analysis)

**Time:** 12-15 minutes

## Quick Win: GDP Nowcast in 2 Minutes

Let's start with a complete working example, then we'll understand how it works.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.dynamic_factor import DynamicFactor
from sklearn.linear_model import LinearRegression
import warnings
warnings.filterwarnings('ignore')

# For this demo, we'll use simulated data that mimics FRED structure
# In production, replace with actual FRED API calls
np.random.seed(42)

# Simulate monthly indicators (2000-2024)
dates = pd.date_range('2000-01-01', '2024-03-01', freq='M')
T = len(dates)

# True latent factor ("economic activity")
factor = np.zeros(T)
for t in range(1, T):
    factor[t] = 0.7 * factor[t-1] + np.random.randn()

# Monthly indicators load on factor + idiosyncratic noise
monthly_data = pd.DataFrame({
    'IP': 0.8 * factor + 0.5 * np.random.randn(T),
    'Employment': 0.7 * factor + 0.6 * np.random.randn(T),
    'Retail_Sales': 0.6 * factor + 0.7 * np.random.randn(T),
    'PMI': 0.5 * factor + 0.8 * np.random.randn(T)
}, index=dates)

# Standardize
monthly_data = (monthly_data - monthly_data.mean()) / monthly_data.std()

# Quarterly GDP (average of factor within quarter + noise)
quarterly_factor = pd.Series(factor, index=dates).resample('Q').mean()
gdp = 2.0 + 1.5 * quarterly_factor + 0.3 * np.random.randn(len(quarterly_factor))
gdp.name = 'GDP_Growth'

print(f"✓ Loaded {len(monthly_data)} months of data ({monthly_data.columns.tolist()})")
print(f"✓ Loaded {len(gdp)} quarters of GDP data")
print(f"\nLatest data points:")
print(monthly_data.tail(3))

In [ ]:
# STEP 1: Estimate Dynamic Factor Model on monthly data
print("Estimating DFM (3 factors, VAR(1) dynamics)...")

dfm = DynamicFactor(
    monthly_data,
    k_factors=3,           # Number of latent factors
    factor_order=1,        # VAR(1) for factor dynamics
    error_cov_type='diagonal'  # Diagonal idiosyncratic errors
)

dfm_res = dfm.fit(maxiter=500, disp=False)
print(f"✓ DFM estimated (log-likelihood: {dfm_res.llf:.1f})")

# Extract filtered factors
factors_monthly = dfm_res.factors.filtered
factors_monthly.columns = [f'Factor_{i+1}' for i in range(3)]

# Aggregate to quarterly
factors_quarterly = factors_monthly.resample('Q').mean()

print(f"✓ Extracted {factors_quarterly.shape[1]} factors")
print(f"\nFactor loadings (how much each indicator loads on factors):")
print(pd.DataFrame(dfm_res.params['loading'], 
                   index=monthly_data.columns, 
                   columns=[f'Factor_{i+1}' for i in range(3)]).round(2))

In [ ]:
# STEP 2: Bridge equation (link quarterly factors to quarterly GDP)
print("\nEstimating bridge equation: GDP = f(Factors)...")

# Align quarterly factors with GDP
common_index = factors_quarterly.index.intersection(gdp.index)
X_bridge = factors_quarterly.loc[common_index]
y_bridge = gdp.loc[common_index]

# Train/test split (use last 4 quarters for testing)
X_train, X_test = X_bridge.iloc[:-4], X_bridge.iloc[-4:]
y_train, y_test = y_bridge.iloc[:-4], y_bridge.iloc[-4:]

# Estimate bridge via OLS
bridge_model = LinearRegression()
bridge_model.fit(X_train, y_train)

print(f"✓ Bridge R² (in-sample): {bridge_model.score(X_train, y_train):.3f}")
print(f"\nBridge coefficients:")
for name, coef in zip(X_bridge.columns, bridge_model.coef_):
    print(f"  {name}: {coef:+.3f}")
print(f"  Intercept: {bridge_model.intercept_:+.3f}")

In [ ]:
# STEP 3: Produce nowcast for most recent quarter
print("\nProducing nowcast for Q1 2024...")

# Current quarter factors (Jan-Mar 2024)
current_quarter_factors = factors_monthly.loc['2024-01':'2024-03'].mean().values.reshape(1, -1)

# Nowcast
nowcast = bridge_model.predict(current_quarter_factors)[0]

# Uncertainty (simplified: residual std from bridge equation)
y_train_pred = bridge_model.predict(X_train)
bridge_std = np.std(y_train - y_train_pred)

print(f"\n{'='*50}")
print(f"  Q1 2024 GDP NOWCAST: {nowcast:.2f}% ± {1.96*bridge_std:.2f}%")
print(f"  (95% confidence interval: [{nowcast-1.96*bridge_std:.2f}, {nowcast+1.96*bridge_std:.2f}])")
print(f"{'='*50}")

## How It Works: Step-by-Step Breakdown

### The Nowcasting Pipeline

```
Monthly Indicators → DFM → Factors → Bridge Equation → GDP Nowcast
   (IP, Emp, ...)     ↓     (r=3)          ↓              ↓
                   Kalman              OLS          Point + Interval
                   Filter
```

In [ ]:
# Visualize the extracted factors
fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

for i in range(3):
    axes[i].plot(factors_monthly.index, factors_monthly.iloc[:, i], 
                 linewidth=1.5, color=f'C{i}')
    axes[i].axhline(0, color='k', linestyle='--', alpha=0.3)
    axes[i].set_ylabel(f'Factor {i+1}', fontsize=11)
    axes[i].grid(alpha=0.3)
    
    # Shade recessions (simplified: just show general pattern)
    recession_periods = [('2001-01', '2001-12'), ('2008-01', '2009-06'), ('2020-02', '2020-04')]
    for start, end in recession_periods:
        axes[i].axvspan(pd.Timestamp(start), pd.Timestamp(end), 
                       alpha=0.2, color='gray', label='Recession' if i==0 else '')

axes[0].set_title('Extracted Latent Factors (Monthly)', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Date', fontsize=11)
axes[0].legend(loc='upper left')
plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("- Factor 1: Primary economic activity (loads heavily on IP, Employment)")
print("- Factor 2: Financial/confidence (captures survey components)")
print("- Factor 3: Consumption/demand (loads on retail sales)")

In [ ]:
# Visualize bridge equation fit
y_bridge_pred = bridge_model.predict(X_bridge)

plt.figure(figsize=(12, 5))
plt.plot(y_bridge.index, y_bridge.values, 'o-', label='Actual GDP', linewidth=2, markersize=4)
plt.plot(y_bridge.index, y_bridge_pred, 's--', label='Bridge Equation Fit', 
         linewidth=2, markersize=4, alpha=0.7)

# Mark out-of-sample period
plt.axvline(y_test.index[0], color='r', linestyle=':', linewidth=2, label='Out-of-Sample →')

plt.ylabel('GDP Growth (%)', fontsize=11)
plt.xlabel('Quarter', fontsize=11)
plt.title('Bridge Equation: Factors → GDP', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Out-of-sample performance
y_test_pred = bridge_model.predict(X_test)
oos_rmse = np.sqrt(np.mean((y_test - y_test_pred)**2))
print(f"\nOut-of-sample RMSE (last 4 quarters): {oos_rmse:.3f}pp")

## Modify This: Experiment with Parameters

Try changing these parameters and observe the impact on nowcast:

In [ ]:
# CUSTOMIZE THESE
N_FACTORS = 3        # Try: 1, 2, 3, 5
FACTOR_LAG_ORDER = 1 # Try: 1, 2, 3
TRAINING_WINDOW = -20  # Use last N quarters for bridge equation (try: -10, -20, -40)

# Re-estimate with your choices
dfm_custom = DynamicFactor(monthly_data, k_factors=N_FACTORS, factor_order=FACTOR_LAG_ORDER)
dfm_custom_res = dfm_custom.fit(maxiter=500, disp=False)

factors_custom = dfm_custom_res.factors.filtered.resample('Q').mean()
X_custom = factors_custom.loc[common_index].iloc[TRAINING_WINDOW:]
y_custom = gdp.loc[common_index].iloc[TRAINING_WINDOW:]

bridge_custom = LinearRegression().fit(X_custom, y_custom)
r2_custom = bridge_custom.score(X_custom, y_custom)

print(f"Results with {N_FACTORS} factors, VAR({FACTOR_LAG_ORDER}):")
print(f"  Bridge R²: {r2_custom:.3f}")
print(f"  DFM log-likelihood: {dfm_custom_res.llf:.1f}")
print(f"\nDid it improve? Compare R² to baseline ({bridge_model.score(X_train, y_train):.3f})")

## News Decomposition: What Changed the Nowcast?

When new data arrives, the nowcast changes. Let's decompose this change into "news" from each indicator.

In [ ]:
# Simulate scenario: March employment comes in stronger than expected
print("Scenario: March employment report just released (+50k above expectations)\n")

# Nowcast BEFORE March employment
monthly_data_feb = monthly_data.loc[:'2024-02'].copy()
dfm_feb = DynamicFactor(monthly_data_feb, k_factors=3, factor_order=1)
dfm_feb_res = dfm_feb.fit(maxiter=500, disp=False)
factors_feb = dfm_feb_res.factors.filtered.resample('Q').mean()
nowcast_feb = bridge_model.predict(factors_feb.iloc[-1].values.reshape(1, -1))[0]

# Nowcast AFTER March employment (already have it: `nowcast`)
nowcast_mar = nowcast

# News impact
news = nowcast_mar - nowcast_feb

print(f"Nowcast evolution:")
print(f"  End of February: {nowcast_feb:.2f}%")
print(f"  After March data: {nowcast_mar:.2f}%")
print(f"  → Revision: {news:+.2f}pp")
print(f"\nInterpretation:")
if news > 0:
    print(f"  March data (especially employment) came in STRONGER than expected.")
    print(f"  This pulled the Q1 GDP nowcast UP by {abs(news):.2f}pp.")
else:
    print(f"  March data came in WEAKER than factor model predicted.")
    print(f"  This pushed the Q1 GDP nowcast DOWN by {abs(news):.2f}pp.")

## Copy-Paste Template: Production Nowcasting Function

Use this in your own projects:

In [ ]:
def nowcast_gdp(monthly_indicators, gdp_history, n_factors=3, factor_order=1):
    """
    Produce GDP nowcast for current quarter.

    Parameters
    ----------
    monthly_indicators : pd.DataFrame (T_month x N)
        Monthly economic indicators (standardized)
    gdp_history : pd.Series (T_quarter)
        Historical quarterly GDP growth
    n_factors : int
        Number of latent factors
    factor_order : int
        VAR lag order for factor dynamics

    Returns
    -------
    nowcast : float
        GDP growth estimate for current quarter
    std_error : float
        Standard error of nowcast
    factors : pd.DataFrame
        Extracted factors (for diagnostics)
    """
    # Estimate DFM
    dfm = DynamicFactor(monthly_indicators, k_factors=n_factors, factor_order=factor_order)
    dfm_res = dfm.fit(maxiter=500, disp=False)
    
    # Extract factors
    factors_monthly = dfm_res.factors.filtered
    factors_quarterly = factors_monthly.resample('Q').mean()
    
    # Bridge equation
    common_idx = factors_quarterly.index.intersection(gdp_history.index)
    X_bridge = factors_quarterly.loc[common_idx]
    y_bridge = gdp_history.loc[common_idx]
    
    bridge = LinearRegression()
    bridge.fit(X_bridge, y_bridge)
    
    # Current quarter nowcast
    current_factors = factors_monthly.iloc[-3:].mean().values.reshape(1, -1)  # Last 3 months
    nowcast = bridge.predict(current_factors)[0]
    
    # Standard error
    residuals = y_bridge - bridge.predict(X_bridge)
    std_error = np.std(residuals)
    
    return nowcast, std_error, factors_monthly

# Example usage
est, se, factors = nowcast_gdp(monthly_data, gdp, n_factors=3)
print(f"\nProduction nowcast: {est:.2f}% ± {1.96*se:.2f}% (95% CI)")

## Go Deeper

**Next steps:**
1. **Real data:** Replace simulated data with actual FRED API calls (see `resources/fred_data_guide.md`)
2. **Ragged edges:** Handle missing recent observations (next notebook)
3. **Evaluation:** Backtest performance over 2000-2024 (notebook 02)
4. **Real-time vintages:** Use ALFRED data for proper evaluation

**Challenge exercises:**
- Add more indicators (housing, credit, sentiment surveys)
- Implement rolling window estimation (update model monthly)
- Compare to NY Fed Nowcast (download their estimates)
- Build uncertainty decomposition (parameter vs shock uncertainty)

**Production deployment:**
- Set up automated FRED data refresh (daily cron job)
- Store nowcast history for revision tracking
- Create dashboard with fan chart visualization
- Alert system for large nowcast changes